In [1]:
import serial
import serial.tools.list_ports
import time
import numpy as np
import plotly
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import pandas as pd
from IPython.display import display, clear_output
from ipywidgets import Button, Output

In [2]:
ports = serial.tools.list_ports.comports()

for port in ports:
    print(port.device, port.manufacturer, port.description)


COM12 FTDI USB Serial Port (COM12)


In [3]:
def decode_packet(packet):
    if len(packet) != 5:
        return "Invalid packet. Packet length does not equals '5'."
    sync, msb, mid, lsb, checksum = packet

    if sync not in [0xA0, 0xA1, 0xA2, 0xA3]:
        return "Invalid packet. Sync byte error."

    if checksum != (msb ^ mid ^ lsb):
        return "Invalid packet. Checksum error."

    value = (msb << 16) | (mid << 8) | lsb    # Combine three bytes to one 24-bit number
    channel = sync - 0xA0 + 1    # Convert sync byte to number of channel

    return channel, value

In [4]:
ports = serial.tools.list_ports.comports()

for port in ports:
    print(port.device, port.manufacturer, port.description)


COM12 FTDI USB Serial Port (COM12)


In [5]:
def decode_packet(packet):
    """Декодирование одного пакета"""
    if len(packet) != 5:
        return None
    
    sync, msb, mid, lsb, checksum = packet
    
    # Проверка синхробайта
    if sync not in [0xA0, 0xA1, 0xA2, 0xA3]:
        return None
    
    # Проверка контрольной суммы
    if checksum != (msb ^ mid ^ lsb):
        return None
    
    # Декодирование значения
    value = (msb << 16) | (mid << 8) | lsb
    channel = sync - 0xA0 + 1
    
    return channel, value


In [6]:
# Параметры
port = "COM12"
baudrate = 115200
dt = 0.01  # интервал обновления
max_points = 1000  # максимальное количество точек на графике


In [7]:
# Создание графиков
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Канал 1', 'Канал 4'),
    vertical_spacing=0.15
)

In [8]:
# Добавление трасс
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=1, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=2, col=1)

fig.update_layout(
    height=800,                     # можно уменьшить высоту
    showlegend=False,
    title_text="Данные с последовательного порта"
)
# Настройка осей
#for i in range(1, 5):
#    fig.update_xaxes(title_text="Время (с)", range=[0, max_points * dt], row=i, col=1)
#    fig.update_yaxes(title_text="Значение", range=[0, 100_000_000], row=i, col=1)


# Настройка осей для двух графиков
for i in range(1, 3):
    fig.update_xaxes(title_text="Время (с)", range=[0, max_points * dt], row=i, col=1)
    fig.update_yaxes(title_text="Значение", row=i, col=1)



In [9]:
# Создание виджета
fig_widget = go.FigureWidget(fig)
display(fig_widget)

print(f"Connecting to port: {port}...")
try:
    ser = serial.Serial(port, baudrate, timeout=1)
    print("Connected!")
except serial.SerialException as e:
    print(f"Error connecting to {port}: {e}")
    exit()

buffer = bytearray()

# Хранение данных для графиков
data_lists = [[], [], [], []]  # список списков для каждого канала
time_list = []  # общий список времени

t = 0.0  # начальное время

try:
    while True:
        # Чтение данных из порта
        if ser.in_waiting:
            raw_data = ser.read(ser.in_waiting)
            buffer.extend(raw_data)
        
        # Обработка буфера
        i = 0
        packets_processed = 0
        
        while i <= len(buffer) - 5:
            if buffer[i] in [0xA0, 0xA1, 0xA2, 0xA3]:
                packet = buffer[i:i+5]
                result = decode_packet(packet)
                if result:
                    channel, value = result
                    #print(f"Channel {channel}: {value}")
                    
                    # Добавляем значение в соответствующий список
                    channel_idx = channel - 1
                    data_lists[channel_idx].append(value)
                    
                    # Ограничиваем размер списка
                    if len(data_lists[channel_idx]) > max_points:
                        data_lists[channel_idx] = data_lists[channel_idx][-max_points:]
                    
                    # Обновляем время
                    time_list.append(t)
                    if len(time_list) > max_points:
                        time_list = time_list[-max_points:]
                    
                    t += dt
                    packets_processed += 1
                    i += 5
                    continue
            i += 1
        
        # Удаляем обработанные данные из буфера
        if i > 0:
            buffer = buffer[i:]
        
        # ОБНОВЛЕНИЕ ГРАФИКОВ - КЛЮЧЕВОЙ МОМЕНТ
        # Создаем новые данные для каждого графика
        for ch in range(4):
            # ОБНОВЛЕНИЕ ГРАФИКОВ - только каналы 1 и 4
            # Канал 1 (индекс 0 в data_lists, трасса 0 в fig_widget)
            if data_lists[0]:
                ch_time = [i * 0.01 for i in range(len(data_lists[0]))]
                fig_widget.data[0].x = ch_time
                fig_widget.data[0].y = data_lists[0]
            
            # Канал 4 (индекс 3 в data_lists, трасса 1 в fig_widget)
            if data_lists[3]:
                ch_time = [i * 0.01 for i in range(len(data_lists[3]))]
                fig_widget.data[1].x = ch_time
                fig_widget.data[1].y = data_lists[3]
        
        # Принудительное обновление виджета (не всегда нужно, но помогает)
        #fig_widget.update_layout(yaxis=(0, max(max(data_lists[ch]))))
        
        # Небольшая пауза для снижения нагрузки CPU
        time.sleep(0.001)
        
except KeyboardInterrupt:
    print("\nStopping...")
except serial.SerialException as se:
    print(f"Serial port error: {se}")
except Exception as e:
    print(f"Error: {e}")
finally:
    ser.close()
    print("Port closed")

# Вывод статистики после остановки
print(f"\nStatistics:")
for i in range(4):
    print(f"Channel {i+1}: {len(data_lists[i])} points")

FigureWidget({
    'data': [{'mode': 'lines',
              'type': 'scatter',
              'uid': '378dd7c4-3ffa-4149-9a57-dda77b5183d8',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'mode': 'lines',
              'type': 'scatter',
              'uid': 'd78d2f81-9134-4a72-856f-b2f79e7cba36',
              'x': [],
              'xaxis': 'x2',
              'y': [],
              'yaxis': 'y2'}],
    'layout': {'annotations': [{'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Канал 1',
                                'x': 0.5,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                          

Connecting to port: COM12...
Connected!

Stopping...
Port closed

Statistics:
Channel 1: 1000 points
Channel 2: 1000 points
Channel 3: 1000 points
Channel 4: 1000 points
